# 🧪 Oxygen T5 Humanizer — Fine-Tuning Notebook

**Fine-tunes the Oxygen T5 model (248M params) to rewrite AI text as natural human text.**

| Step | Task | Time (T4 GPU) |
|------|------|---------------|
| 1 | Install dependencies | ~2 min |
| 2 | Upload model from Google Drive | ~1 min |
| 3 | Load HuggingFace datasets (HC3 + Wiki + dmitva) | ~3 min |
| 4 | Prepare training pairs | ~1 min |
| 5 | Fine-tune T5 on GPU | ~15-40 min |
| 6 | Test fine-tuned model | ~2 min |
| 7 | Download fine-tuned model | ~2 min |

**Total: ~25-50 min on a T4 GPU**

### Prerequisites
1. Upload `oxygen-model/` folder to Google Drive (any location)
2. Use **GPU runtime**: Runtime → Change runtime type → T4 GPU

---

## Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install -q torch transformers datasets safetensors tqdm accelerate

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\u26a0\ufe0f No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    print("Training on CPU will be extremely slow.")

## Step 2: Upload Model from Google Drive

Upload your `oxygen-model/` folder to Google Drive first, then run this cell.

**Expected files in the folder:**
- `model.safetensors` (~945 MB)
- `config.json`
- `tokenizer.json`
- `tokenizer_config.json`
- `generation_config.json`

In [ ]:
from google.colab import drive
import os, shutil, time

# Mount Drive (force remount to avoid stale connections)
drive.mount('/content/drive', force_remount=True)

# ====================================================================
# EDIT THIS PATH to match where you put oxygen-model/ in Google Drive
# ====================================================================
DRIVE_MODEL_PATH = "/content/drive/MyDrive/oxygen-model"

LOCAL_MODEL_DIR = "/content/oxygen-model"

def copy_with_retry(src, dst, max_retries=3):
    """Copy a single file with retries (handles Drive flakiness)."""
    for attempt in range(max_retries):
        try:
            shutil.copy2(src, dst)
            return True
        except Exception as e:
            print(f"  Retry {attempt+1}/{max_retries} for {os.path.basename(src)}: {e}")
            time.sleep(2)
            if attempt == max_retries - 1:
                raise
    return False

if os.path.exists(DRIVE_MODEL_PATH):
    print(f"Copying model from Drive: {DRIVE_MODEL_PATH}")
    if os.path.exists(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

    # Copy files one at a time (more reliable for large files)
    src_files = os.listdir(DRIVE_MODEL_PATH)
    for f in src_files:
        src = os.path.join(DRIVE_MODEL_PATH, f)
        dst = os.path.join(LOCAL_MODEL_DIR, f)
        if os.path.isfile(src):
            size_mb = os.path.getsize(src) / 1e6
            print(f"  Copying {f} ({size_mb:.1f} MB)...")
            copy_with_retry(src, dst)

    files = os.listdir(LOCAL_MODEL_DIR)
    for f in files:
        size = os.path.getsize(os.path.join(LOCAL_MODEL_DIR, f))
        print(f"  \u2713 {f}: {size/1e6:.1f} MB")
    assert "model.safetensors" in files, "model.safetensors not found!"
    assert "config.json" in files, "config.json not found!"
    print("\n\u2705 Model loaded successfully!")
else:
    print(f"\u274c Model not found at: {DRIVE_MODEL_PATH}")
    print("\nPlease upload the oxygen-model/ folder to Google Drive and update DRIVE_MODEL_PATH above.")
    print("\nExpected structure in Drive:")
    print("  MyDrive/oxygen-model/model.safetensors")
    print("  MyDrive/oxygen-model/config.json")
    print("  MyDrive/oxygen-model/tokenizer.json")
    print("  MyDrive/oxygen-model/tokenizer_config.json")

## Step 3: Load Model & Verify

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

MODEL_DIR = "/content/oxygen-model"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

print("Loading model...")
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_DIR, local_files_only=True, torch_dtype=torch.float32
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"\n\u2705 Model loaded: {n_params:,} params on {device}")
print(f"   Architecture: d_model={model.config.d_model}, layers={model.config.num_layers}, heads={model.config.num_heads}")

# Quick generation test
test_in = "Artificial intelligence has fundamentally transformed modern technology."
inputs = tokenizer(test_in, return_tensors="pt", max_length=256, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, num_beams=4, do_sample=False,
                         no_repeat_ngram_size=3, early_stopping=True)
test_out = tokenizer.decode(out[0], skip_special_tokens=True)
print(f"\nTest IN:  {test_in}")
print(f"Test OUT: {test_out}")

## Step 4: Load Training Data

Combines:
- **6 local AI/Human text pairs** (embedded below, split into ~38 sentence pairs)
- **HC3** — Human vs ChatGPT answers (~3,300 pairs)
- **GPT-Wiki-Intro** — Wikipedia vs GPT intros (~3,300 pairs)
- **dmitva** — AI vs human labeled texts (~85 pairs)

Total: ~7,000+ training pairs

In [ ]:
import re
import random
from datasets import load_dataset

def split_sentences(text):
    """Split text into sentences."""
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [s.strip() for s in parts if s.strip()]

# ── Local curated pairs (embedded) ──
# RULES: Human targets must be formal academic prose.
# No contractions, no first person, no rhetorical questions.
# Just naturally flowing text with AI-tell markers removed.
LOCAL_PAIRS_RAW = [
    # Sample 1: AI/NLP
    (
        "The intersection of artificial intelligence and natural language processing has become a focal point of contemporary research. Furthermore, the implications of these technological advances extend far beyond the technical realm. Moreover, society faces unprecedented challenges in distinguishing authentic human discourse from machine-generated content. In conclusion, the urgency to develop robust detection mechanisms cannot be overstated. It is important to note that the current landscape presents both opportunities and risks. Additionally, stakeholders across academia and industry must collaborate to address these emerging concerns. Consequently, future research directions will undoubtedly focus on improving detection accuracy and understanding the underlying mechanisms of language generation.",
        "Artificial intelligence and natural language processing now intersect in ways that carry significant implications beyond the technical domain. Society faces a growing challenge in distinguishing authentic human writing from machine-generated text, and the urgency of developing reliable detection tools has become difficult to overstate. The current landscape presents both opportunities and risks, requiring collaboration across academia and industry. Future research will likely center on improving detection accuracy while deepening understanding of how language generation systems operate."
    ),
    # Sample 2: Contemporary literature
    (
        "Contemporary literature, defined as works produced within the last few decades, encompasses a diverse array of genres and thematic preoccupations. The evolution of narrative structures has been documented extensively by literary scholars. Furthermore, the emergence of postmodern techniques has fundamentally altered reader expectations and interpretive frameworks. Moreover, authors such as David Foster Wallace and Don DeLillo have pioneered innovative approaches to storytelling. To summarize, the trajectory of contemporary literature reflects broader cultural anxieties and technological transformations. In fact, this phenomenon warrants comprehensive examination from multiple theoretical perspectives. Additionally, the influence of digital media on literary production cannot be dismissed. Consequently, future generations of scholars must grapple with these evolving aesthetic paradigms.",
        "Literature produced in recent decades spans a wide range of genres and thematic concerns, with narrative structures evolving considerably. Postmodern techniques have reshaped reader expectations and interpretive frameworks, as demonstrated by authors like David Foster Wallace and Don DeLillo, who pioneered new approaches to storytelling. These literary shifts reflect broader cultural anxieties tied to meaning, identity, and technology. Digital media has also exerted a transformative influence on literary production. Scholars engaging with this period must contend with evolving aesthetic paradigms that resist traditional modes of analysis."
    ),
    # Sample 3: Climate policy
    (
        "The efficacy of climate change mitigation strategies has become a subject of considerable debate within scientific and policy circles. Multiple linear regression analyses have demonstrated correlations between carbon emissions and atmospheric temperature fluctuations. It is particularly noteworthy that international agreements such as the Paris Accord attempt to regulate greenhouse gas emissions systematically. Furthermore, empirical data indicates that renewable energy adoption has increased substantially over the past decade. Nevertheless, transition challenges persist in fossil fuel dependent economies. Additionally, technological innovations in carbon capture mechanisms present promising alternatives. To elaborate further, the multifaceted nature of this challenge necessitates interdisciplinary collaboration. In conclusion, comprehensive evaluation of policy effectiveness remains essential for informed decision-making.",
        "Climate change mitigation strategies remain a subject of debate across scientific and policy circles. Regression analyses have established correlations between carbon emissions and temperature fluctuations, and international agreements such as the Paris Accord seek to regulate greenhouse gas output on a systematic basis. Renewable energy adoption has grown substantially over the past decade, though fossil fuel dependent economies continue to face transition challenges. Carbon capture technologies offer a promising complement, but no single approach is sufficient on its own. Effective policy evaluation requires interdisciplinary collaboration and sustained attention to economic incentives alongside technological development."
    ),
    # Sample 4: Cognitive linguistics
    (
        "Cognitive linguistics provides a theoretical framework for understanding metaphorical language acquisition in second language learners. Research has consistently demonstrated that metaphors function as conceptual systems rather than mere stylistic devices. Moreover, cross-linguistic studies indicate that metaphorical mappings exhibit both universal and language-specific characteristics. Furthermore, the role of embodied cognition in generating metaphors has been substantiated through neuroscientific evidence. In particular, neuroimaging data reveals activation patterns consistent with embodied metaphor theory. Additionally, pedagogical applications of cognitive linguistic principles have yielded improved learning outcomes. To summarize, the integration of cognitive approaches with traditional language instruction methodologies represents a significant paradigm shift. It is important to acknowledge that further investigation remains requisite for optimal classroom implementation.",
        "Cognitive linguistics offers a useful framework for examining metaphorical language acquisition among second language learners. Research consistently shows that metaphors operate as conceptual systems rather than surface-level stylistic devices, and cross-linguistic studies reveal both universal and language-specific characteristics in metaphorical mappings. Neuroscientific evidence supports the role of embodied cognition in metaphor processing, with neuroimaging data showing activation patterns consistent with embodied metaphor theory. Pedagogical applications of these principles have produced improved learning outcomes, though further investigation is needed to optimize classroom implementation."
    ),
    # Sample 5: Organizational behavior
    (
        "Organizational behavior scholarship has extensively documented the phenomenon of employee engagement and its correlative relationship with productivity metrics. Empirical investigations utilizing Likert-scale questionnaires have revealed statistically significant differences in performance outcomes between engaged and disengaged workforce segments. Furthermore, longitudinal studies demonstrate that workplace culture interventions produce measurable improvements in retention rates. Moreover, the role of transformational leadership in facilitating organizational commitment has been validated across multiple organizational contexts. In fact, meta-analytical synthesis of extant literature indicates effect sizes of considerable magnitude. Additionally, psychological safety\u2014operationalized as perceived interpersonal risk-taking capacity\u2014emerges as a critical mediating variable. To conclude, the cumulative evidence supports implementation of evidence-based human resources practices.",
        "Employee engagement has a well-documented relationship with productivity, and empirical studies using Likert-scale instruments reveal statistically significant performance differences between engaged and disengaged workers. Longitudinal research shows that workplace culture interventions produce measurable gains in retention, while transformational leadership has been validated as a driver of organizational commitment across multiple settings. Meta-analyses of the existing literature indicate substantial effect sizes. Psychological safety, defined as the perceived capacity for interpersonal risk-taking, emerges as a critical mediating variable. Taken together, the evidence supports the adoption of evidence-based human resources practices."
    ),
    # Sample 6: Consciousness studies
    (
        "The phenomenological approach to consciousness studies represents a paradigmatic departure from traditional reductionist frameworks. Phenomenologists contend that subjective experience possesses ontological primacy and cannot be adequately characterized through physicalist reduction alone. Furthermore, the concept of intentionality\u2014understood as consciousness's directedness toward objects\u2014constitutes a fundamental feature of phenomenal experience. Moreover, recent developments in neurophenomenology attempt to reconcile first-person and third-person perspectives. It should be noted that this reconciliation remains theoretically contested. Additionally, contemplative research methodologies, informed by Buddhist philosophical traditions, offer innovative empirical approaches. In conclusion, contemporary consciousness studies increasingly recognizes the necessity of phenomenological investigation alongside neuroscientific inquiry.",
        "Phenomenology represents a departure from reductionist frameworks in consciousness studies, holding that subjective experience cannot be fully characterized through physicalist reduction alone. Intentionality, the directedness of consciousness toward objects, remains a central concept in phenomenal experience. Neurophenomenology has emerged as an attempt to bridge first-person and third-person perspectives, though this reconciliation remains contested. Contemplative research methodologies, particularly those drawing on Buddhist philosophical traditions, have introduced new empirical approaches to the field. Contemporary consciousness studies increasingly recognizes the value of phenomenological investigation conducted alongside neuroscientific inquiry."
    ),
    # Sample 7: Government overreach — Introduction
    (
        "The role of government in society has long been debated, particularly regarding the balance between necessary intervention and excessive control. The distinction between \"government overreach\" and \"responsible government\" is central to understanding how power should be exercised in democratic systems. While governments are expected to create policies that promote public welfare, protect rights, and regulate markets, there is always a risk that such authority may extend beyond appropriate limits. This paper discusses the meaning of government overreach and responsible government, explores their differences, and applies these concepts to key public policy areas to demonstrate their relevance in contemporary governance.",
        "The involvement of the government in the society is an issue that has been argued over and especially in terms of the need to intervene and overdoing it. The main measure in understanding how power is to be exercised in democratic systems is the distinction between government overreach and responsible government. Although governments have the duty to develop policies that enhance the common good, safeguard rights and control markets, there is always a chance that this power can be pushed beyond reasonable boundaries. This paper explains what is meant by government overreach and responsible government, the difference between the two, and applies both to major areas of public policy to illustrate how the two concepts apply in the modern day governance."
    ),
    # Sample 8: Government overreach — Definition
    (
        "Government overreach refers to situations where authorities exceed their legitimate powers, often interfering excessively in the lives of citizens, institutions, or markets. It typically involves the misuse or expansion of power beyond constitutional or ethical boundaries, potentially undermining individual freedoms and institutional balance. For example, excessive executive control or disregard for checks and balances can weaken democratic systems (Acharya & Hanspal, 2025). Similarly, judicial overreach occurs when courts go beyond their interpretative role and begin to make policy decisions, thereby interfering with legislative or executive functions (Van Heerden, 2018).",
        "Government overreach is the term used to refer to cases in which the government abuses its legitimate powers and tends to meddle in the lives of the citizens, institutions or the market. It is generally the abuse or misuse of power or in any way it extends to the constitutional or ethical limit that may compromise the liberty of individuals and balances of an institution. In the case of excessive executive power, or neglect of checks and balances, the democratic systems can be weakened (Acharya & Hanspal, 2025). Likewise, judicial overreach is the situation when courts exceed their interpretative authority and start to engage in a policy-making process, thus intruding on legislative or executive operations (Van Heerden, 2018)."
    ),
    # Sample 9: Responsible government — Definition
    (
        "On the other hand, responsible government refers to the appropriate and accountable use of authority to serve public interests. It involves balancing intervention with respect for individual rights, institutional boundaries, and democratic principles. According to Taylor (2023), responsible government requires aligning public policies with societal needs while ensuring that actions remain transparent, ethical, and proportionate. This concept emphasizes accountability, responsiveness, and the careful use of power to address social and economic challenges.",
        "Responsible government on the other hand is the proper and accountable exercise of power to the benefit of the people. It is about balancing intervention and consideration of individual rights, institutional boundaries and democratic principles. Taylor (2023) points out that responsible government must involve balancing the societal needs with the policies of the government and that the actions must be transparent, ethical, and proportionate. This idea highlights responsibility, responsiveness and cautiousness in using power to deal with social and economic problems."
    ),
    # Sample 10: Overreach vs responsibility — Key difference
    (
        "The key difference between these two concepts lies in balance and accountability. Responsible government operates within defined limits and prioritizes public welfare, while government overreach involves exceeding those limits, often leading to unintended or harmful consequences. As Segaro and Haag (2022) note, even well-intentioned interventions can produce negative outcomes when governments fail to consider broader stakeholder impacts or impose excessive control.",
        "Balance and accountability is the main difference between the two concepts. Government responsibility is within set boundaries and is focused on the welfare of the people, whereas government overreach refers to going beyond those boundaries, and usually has unintentional or damageable effects. Even with the best intentions, interventions can have adverse effects, as Segaro and Haag (2022) observe that when governments do not take into account the wider effects of the stakeholders involved and overexert control, such interventions can lead to negative results."
    ),
    # Sample 11: Risks — Separation of powers
    (
        "Government overreach can have significant implications for democracy, governance, and society. One major risk is the erosion of the separation of powers. When one branch of government, particularly the executive, dominates decision-making, it undermines institutional checks and balances designed to prevent abuse of power (Acharya & Hanspal, 2025). This imbalance can lead to authoritarian tendencies and reduced accountability.",
        "The consequences of government overreach can be very dramatic on democracy, governance and society. The loss of the separation of powers is one of the threats. The dominance of one arm of government especially the executive in the decision-making process is harmful to institutional checks and balances aimed at deterring abuse of power (Acharya & Hanspal, 2025). Such imbalance may result in dictatorial nature and lack of accountability."
    ),
    # Sample 12: Risks — Individual freedoms
    (
        "Another consequence is the restriction of individual freedoms. Policies that excessively regulate speech, behavior, or economic activity can infringe on fundamental rights. Boland (2013) highlights the tension between free speech and government regulation, emphasizing that excessive control over expression can undermine democratic values and human dignity. In such cases, government actions intended to maintain order may instead suppress legitimate freedoms.",
        "Other effects include limiting individual liberties. Much speech, behavior, or economic activity regulation can violate core rights. Boland (2013) points out the conflict between freedom of expression and government regulation by noting that when the government controls expression too much, it destroys democratic principles and human dignity. When this happens, the government interventions aimed at upholding order can end up oppressing the legitimate freedoms."
    ),
    # Sample 13: Risks — Ineffective policies
    (
        "Additionally, overreach can result in ineffective or counterproductive policies. Segaro and Haag (2022) demonstrate that government interventions, even when designed to achieve positive outcomes, can fail when they overlook stakeholder dynamics or impose rigid structures. This often leads to inefficiencies, resistance from affected groups, and unintended negative consequences.",
        "Also overreach may lead to ineffective or counter-productive policies. Segaro and Haag (2022) show that even interventions with positive intentions by the government may fail due to their failure to consider stakeholder dynamics or to introduce strict structures. This has a tendency to cause inefficiencies, opposition by the groups that are affected and undesired adverse effects."
    ),
    # Sample 14: Responsible government — Characteristics
    (
        "In contrast, responsible government is characterized by accountability, transparency, and respect for institutional boundaries. It involves making decisions that are evidence-based, inclusive, and aligned with democratic principles. Governments practicing responsibility ensure that policies are proportional to the issues they address and that citizens' rights are protected.",
        "Accountability, transparency and respectful boundaries of institutions on the other hand define responsible government. It entails the process of coming up with evidence-based, inclusive, and democratic decisions. Responsible governments make sure that the policies are commensurate to the problems that they are tackling and that the rights of citizens are not compromised."
    ),
    # Sample 15: Responsible government — Trust
    (
        "Responsible government also promotes trust and legitimacy. When citizens perceive that their government acts in their best interests and respects their rights, they are more likely to support policies and comply with regulations. This trust is essential for effective governance and long-term stability.",
        "Trust and legitimacy are also encouraged by responsible government. Once citizens are convinced that the government is doing what is in their best interest and respecting their rights, they will tend to support policies and also follow regulations. This confidence is critical towards good governance and stability."
    ),
    # Sample 16: Application — Public health
    (
        "In the area of public health, government intervention is often necessary to protect populations from disease and promote well-being. However, excessive control or influence can raise concerns about bias and overreach. Senior et al. (2025) explain that governments play a significant role in shaping public health research and policy, which can be beneficial but also problematic if it leads to undue influence or suppression of alternative perspectives. Responsible government in this context involves supporting research and implementing policies based on evidence while maintaining transparency and independence.",
        "Government intervention in the field of public health is usually needed to prevent populations against the disease and ensure well being. Nevertheless, control or influence too much can give rise to doubt about prejudice and overreach. According to Senior et al. (2025), governments have a strong role in influencing the research and policy of the public, which might be useful but also troublesome in case of the excessive influence or suppression of other opinion. The role of responsible government in this regard is supporting the research and implementing policies based on evidence and being transparent and independent."
    ),
    # Sample 17: Application — Economic regulation
    (
        "Economic regulation is another area where the balance between overreach and responsibility is critical. Governments must regulate markets to prevent exploitation and ensure fairness, but excessive intervention can stifle innovation and economic growth. Segaro and Haag (2022) illustrate how poorly designed interventions can disrupt stakeholder relationships and lead to unintended negative outcomes. Responsible governance requires creating policies that support economic stability without imposing unnecessary restrictions.",
        "Another field of economic regulation is an area where overreach and responsibility are crucial. To avoid exploitation and in favor of equity, governments need to control markets; however, over control kills innovation and economic development. Segaro and Haag (2022) demonstrate the impact of poorly designed interventions that can destabilize the relationship between stakeholders and cause unwanted adverse consequences. The need to have responsible governance would involve formulation of policies that would facilitate economic stability but not overly restrictive."
    ),
    # Sample 18: Application — Civil liberties
    (
        "Civil liberties, particularly freedom of speech, also demonstrate the importance of this distinction. While governments may regulate speech to prevent harm, excessive restrictions can undermine democratic values. Boland (2013) argues that maintaining a balance between regulation and freedom is essential to preserving human dignity and democratic governance. Responsible government ensures that any limitations on rights are justified, proportionate, and consistent with constitutional principles.",
        "This difference is also important in civil liberties, especially the freedom of speech. Although the governments can control speech to avoid any harm, too much control can demolish the democratic principles. According to Boland (2013), a balance between control and liberty is necessary to uphold human dignity and democratic rule. Good government keeps the restrictions of rights legitimate, proportional and in line with constitutional principles."
    ),
    # Sample 19: Conclusion
    (
        "The difference between government overreach and responsible government lies in the extent and manner in which power is exercised. While overreach involves exceeding legitimate authority and potentially harming democratic principles, responsible government emphasizes balance, accountability, and the protection of public interests. This distinction is crucial in various public policy areas, including public health, economic regulation, civil liberties, and institutional governance. By maintaining appropriate limits and prioritizing transparency and responsiveness, governments can effectively serve their citizens while avoiding the risks associated with excessive intervention. Ultimately, achieving this balance is essential for sustaining democratic systems and promoting long-term societal well-being.",
        "The distinction between government overreach and responsible government is the extent and nature of exercise of power. Whereas overreach is a case of pushing boundaries, which may be detrimental to democratic values, responsible government highlights moderation, accountability, and safeguarding of the common good. This difference is vital in many aspects of public policy, such as public health, economic regulation, civil liberties, and institutional governance. Governments can serve their citizens effectively without taking the risks of overly intervening since proper limits can be maintained and transparency and responsiveness can be prioritized. Finally, such a balance is the key to maintaining the democratic systems and ensuring the long-term welfare of the society."
    ),
]

# Build local pairs (full-text + sentence-level)
local_pairs = []
for ai_text, human_text in LOCAL_PAIRS_RAW:
    local_pairs.append((ai_text, human_text))
    ai_sents = split_sentences(ai_text)
    human_sents = split_sentences(human_text)
    if abs(len(ai_sents) - len(human_sents)) <= 2:
        for a, h in zip(ai_sents, human_sents):
            if len(a.split()) >= 5 and len(h.split()) >= 5:
                local_pairs.append((a, h))

print(f"Local curated pairs: {len(local_pairs)}")

In [ ]:
# ── HuggingFace datasets ──
# CRITICAL: HC3 and GPT-Wiki pairs are NOT rewrites — they're independently
# written texts on the same topic. We MUST filter to only keep pairs with
# significant word overlap, otherwise the model learns to output gibberish.

MAX_PER_DATASET = 3500
hf_pairs_raw = []

def word_overlap(text_a, text_b):
    """Calculate Jaccard word overlap between two texts."""
    a = {w.lower().strip(".,;:!?\"'()[]") for w in text_a.split() if len(w) > 2}
    b = {w.lower().strip(".,;:!?\"'()[]") for w in text_b.split() if len(w) > 2}
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

# --- HC3: Human ChatGPT Comparison ---
print("Loading HC3 dataset...")
try:
    hc3 = load_dataset("Hello-SimpleAI/HC3", "all", split="train")
    count = 0
    for item in hc3:
        if count >= MAX_PER_DATASET:
            break
        human_answers = item.get("human_answers", [])
        chatgpt_answers = item.get("chatgpt_answers", [])
        if human_answers and chatgpt_answers:
            ai_text = chatgpt_answers[0].strip()
            human_text = human_answers[0].strip()
            if 20 <= len(ai_text.split()) <= 200 and 20 <= len(human_text.split()) <= 200:
                hf_pairs_raw.append((ai_text, human_text))
                count += 1
    print(f"  HC3: {count} raw pairs")
except Exception as e:
    print(f"  HC3 load_dataset failed: {e}")
    print("  Trying HC3 via direct JSONL download...")
    try:
        import json as _json
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("Hello-SimpleAI/HC3", "all.jsonl", repo_type="dataset")
        count = 0
        with open(path, encoding="utf-8") as f:
            for line in f:
                if count >= MAX_PER_DATASET:
                    break
                item = _json.loads(line)
                human_answers = item.get("human_answers", [])
                chatgpt_answers = item.get("chatgpt_answers", [])
                if human_answers and chatgpt_answers:
                    ai_text = chatgpt_answers[0].strip()
                    human_text = human_answers[0].strip()
                    if 20 <= len(ai_text.split()) <= 200 and 20 <= len(human_text.split()) <= 200:
                        hf_pairs_raw.append((ai_text, human_text))
                        count += 1
        print(f"  HC3 (JSONL fallback): {count} raw pairs")
    except Exception as e2:
        print(f"  HC3 fallback also failed: {e2}")

# --- GPT-Wiki-Intro ---
print("Loading GPT-Wiki-Intro dataset...")
try:
    wiki = load_dataset("aadityaubhat/GPT-wiki-intro", split="train")
    count = 0
    for item in wiki:
        if count >= MAX_PER_DATASET:
            break
        ai_text = item.get("generated_intro", "").strip()
        human_text = item.get("wiki_intro", "").strip()
        if 20 <= len(ai_text.split()) <= 200 and 20 <= len(human_text.split()) <= 200:
            hf_pairs_raw.append((ai_text, human_text))
            count += 1
    print(f"  GPT-Wiki-Intro: {count} raw pairs")
except Exception as e:
    print(f"  GPT-Wiki-Intro failed: {e}")

# --- dmitva: AI vs Human ---
print("Loading dmitva dataset...")
try:
    dmitva = load_dataset("dmitva/human_ai_generated_text", split="train")
    print(f"  dmitva columns: {dmitva.column_names}")

    ai_texts, human_texts = [], []
    text_col = "text" if "text" in dmitva.column_names else dmitva.column_names[0]
    label_col = "label" if "label" in dmitva.column_names else dmitva.column_names[-1]

    for item in dmitva:
        text = str(item.get(text_col, "")).strip()
        label = item.get(label_col, -1)
        if isinstance(label, str):
            label = 1 if label.lower() in ("ai", "generated", "machine", "1") else 0
        if 20 <= len(text.split()) <= 200:
            if label == 1:
                ai_texts.append(text)
            elif label == 0:
                human_texts.append(text)

    ai_texts.sort(key=lambda x: len(x.split()))
    human_texts.sort(key=lambda x: len(x.split()))
    n = min(len(ai_texts), len(human_texts), MAX_PER_DATASET)
    for i in range(n):
        hf_pairs_raw.append((ai_texts[i], human_texts[i]))
    print(f"  dmitva: {n} pairs (ai_pool={len(ai_texts)}, human_pool={len(human_texts)})")
except Exception as e:
    print(f"  dmitva failed: {e}")

print(f"\nRaw HuggingFace pairs: {len(hf_pairs_raw)}")

# ── CRITICAL FILTER: Only keep pairs that are actual rewrites ──
# Pairs with <15% word overlap are NOT rewrites — they'll poison the model.
# Pairs with >85% overlap are too similar to learn anything from.
MIN_OVERLAP = 0.15
MAX_OVERLAP = 0.85
hf_pairs = []
overlap_stats = {"too_low": 0, "too_high": 0, "good": 0}

for ai_text, human_text in hf_pairs_raw:
    ov = word_overlap(ai_text, human_text)
    if ov < MIN_OVERLAP:
        overlap_stats["too_low"] += 1
    elif ov > MAX_OVERLAP:
        overlap_stats["too_high"] += 1
    else:
        hf_pairs.append((ai_text, human_text))
        overlap_stats["good"] += 1

print(f"\n🔍 Word overlap filter (Jaccard {MIN_OVERLAP:.0%}–{MAX_OVERLAP:.0%}):")
print(f"   ✅ Kept: {overlap_stats['good']} pairs")
print(f"   ❌ Too different (<{MIN_OVERLAP:.0%} overlap): {overlap_stats['too_low']}")
print(f"   ⚠️ Too similar (>{MAX_OVERLAP:.0%} overlap): {overlap_stats['too_high']}")
print(f"\nFiltered HuggingFace pairs: {len(hf_pairs)}")

In [ ]:
# ── Combine & split ──
# Curated pairs are HIGH quality — repeat them so they're not drowned out
# by the noisier HuggingFace pairs.

CURATED_REPEAT = 5  # each curated pair appears 5x in training
boosted_local = local_pairs * CURATED_REPEAT
all_pairs = boosted_local + hf_pairs
random.shuffle(all_pairs)

val_size = max(10, int(len(all_pairs) * 0.1))
val_pairs = all_pairs[:val_size]
train_pairs = all_pairs[val_size:]

print(f"\n✅ Final dataset:")
print(f"   Curated pairs: {len(local_pairs)} × {CURATED_REPEAT} = {len(boosted_local)}")
print(f"   HuggingFace pairs: {len(hf_pairs)}")
print(f"   Train: {len(train_pairs)} pairs")
print(f"   Val:   {len(val_pairs)} pairs")
print(f"   Total: {len(all_pairs)} pairs")

# Preview a few pairs
print("\n── Sample pairs ──")
for i in range(min(3, len(train_pairs))):
    ai, human = train_pairs[i]
    ov = word_overlap(ai, human)
    print(f"\nPair {i+1} (overlap: {ov:.0%}):")
    print(f"  AI:    {ai[:100]}...")
    print(f"  Human: {human[:100]}...")

## Step 5: Fine-Tune the T5 Model

Training config (tuned to prevent catastrophic forgetting):
- **Learning rate:** 1e-5 (conservative — prevents model collapse)
- **Batch size:** 4 with gradient accumulation 4 (effective batch = 16)
- **Epochs:** 3 (increase to 5 for better quality)
- **Warmup:** 300 steps
- **Max input/target:** 256 tokens
- **Curated pair boost:** 5× repetition to outweigh noisy HF pairs

Adjust `EPOCHS` and `BATCH_SIZE` below if needed.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import time

# ============================
# TRAINING HYPERPARAMETERS
# ============================
EPOCHS = 3              # 3 is good, 5 is better if you have time
BATCH_SIZE = 4          # 4 for safer training (less gradient noise)
LEARNING_RATE = 1e-5    # LOWERED from 5e-5 — prevents catastrophic forgetting
GRAD_ACCUM = 4          # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
WARMUP_STEPS = 300      # longer warmup for stability
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 256
MAX_GRAD_NORM = 1.0
SAVE_EVERY = 500        # save checkpoint every N optimizer steps
# ============================


class HumanizerDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input=MAX_INPUT_LEN, max_target=MAX_TARGET_LEN):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ai_text, human_text = self.pairs[idx]
        input_enc = self.tokenizer(ai_text, max_length=self.max_input, truncation=True,
                                   padding="max_length", return_tensors="pt")
        target_enc = self.tokenizer(human_text, max_length=self.max_target, truncation=True,
                                    padding="max_length", return_tensors="pt")
        input_ids = input_enc["input_ids"].squeeze(0)
        attention_mask = input_enc["attention_mask"].squeeze(0)
        labels = target_enc["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


# Build datasets
train_dataset = HumanizerDataset(train_pairs, tokenizer)
val_dataset = HumanizerDataset(val_pairs, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS

# Optimizer
no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
param_groups = [
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay) and p.requires_grad],
     "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay) and p.requires_grad],
     "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, total_steps)

print(f"Training config:")
print(f"  Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")
print(f"  Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR: {LEARNING_RATE}, Warmup: {WARMUP_STEPS}, Total steps: {total_steps}")
print(f"  Device: {device}")
print(f"\nStarting training...\n")

In [ ]:
# ── Training Loop ──

import os, time
os.makedirs("/content/checkpoints", exist_ok=True)
best_val_loss = float("inf")
train_losses = []
val_losses = []
global_step = 0
start_time = time.time()

model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
    for step, batch in enumerate(progress):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_loss += outputs.loss.item()
        num_batches += 1

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            progress.set_postfix(
                loss=f"{outputs.loss.item():.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
                step=global_step,
            )

            # Save checkpoint
            if SAVE_EVERY > 0 and global_step % SAVE_EVERY == 0:
                ckpt_path = f"/content/checkpoints/checkpoint-{global_step}"
                model.save_pretrained(ckpt_path)
                tokenizer.save_pretrained(ckpt_path)
                print(f"\n  Checkpoint saved: {ckpt_path}")

    avg_train = epoch_loss / max(num_batches, 1)
    train_losses.append(avg_train)

    # Validation
    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            val_batches += 1
    avg_val = val_loss / max(val_batches, 1)
    val_losses.append(avg_val)
    model.train()

    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{EPOCHS} — train_loss: {avg_train:.4f}, val_loss: {avg_val:.4f}, time: {elapsed/60:.1f}m")

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained("/content/checkpoints/best")
        tokenizer.save_pretrained("/content/checkpoints/best")
        print(f"  ⭐ New best model (val_loss={avg_val:.4f})")

total_time = time.time() - start_time
print(f"\n✅ Training complete in {total_time/60:.1f} minutes")
print(f"   Best val loss: {best_val_loss:.4f}")
print(f"   Global steps: {global_step}")

# Save metadata so plot cell can recover after runtime restart
import json as _json
_meta = {"train_losses": train_losses, "val_losses": val_losses,
         "best_val_loss": best_val_loss, "global_step": global_step}
with open("/content/checkpoints/best/training_metadata.json", "w") as _f:
    _json.dump(_meta, _f)
print("   Metadata saved to checkpoint.")

In [ ]:
# ── Plot training curves ──

import matplotlib.pyplot as plt
import json as _json

# Recover losses from saved metadata if variables were lost (runtime restart)
if 'train_losses' not in dir() or 'val_losses' not in dir():
    print("⚠️ train_losses/val_losses not in memory — loading from checkpoint metadata...")
    _meta_path = "/content/checkpoints/best/training_metadata.json"
    _meta_path2 = "/content/oxygen-model-finetuned/training_metadata.json"
    for _p in [_meta_path, _meta_path2]:
        if __import__('os').path.exists(_p):
            with open(_p) as _f:
                _meta = _json.load(_f)
            train_losses = _meta["train_losses"]
            val_losses = _meta["val_losses"]
            print(f"  ✅ Loaded from {_p}")
            break
    else:
        # Last resort: check if training loop saved them
        print("  ❌ No metadata file found. Re-run the training loop cell first.")
        train_losses, val_losses = [], []

if train_losses and val_losses:
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    epochs_range = range(1, len(train_losses) + 1)
    ax.plot(epochs_range, train_losses, 'b-o', label='Train Loss')
    ax.plot(epochs_range, val_losses, 'r-o', label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Fine-Tuning Loss Curves')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150)
    plt.show()
    print("Saved: /content/training_curves.png")
else:
    print("No loss data to plot.")

## Step 6: Test the Fine-Tuned Model

Generates humanized outputs for several AI-style test texts and compares with the original model.

In [ ]:
# Load the best checkpoint
print("Loading best checkpoint...")
best_model = T5ForConditionalGeneration.from_pretrained(
    "/content/checkpoints/best", torch_dtype=torch.float32
).to(device)
best_model.eval()

# Also load original for comparison
print("Loading original model for comparison...")
orig_model = T5ForConditionalGeneration.from_pretrained(
    "/content/oxygen-model", local_files_only=True, torch_dtype=torch.float32
).to(device)
orig_model.eval()

test_texts = [
    "Artificial intelligence has fundamentally transformed the landscape of modern technology.",
    "Furthermore, the implications of these advances extend far beyond the technical realm.",
    "It is important to note that current detection mechanisms remain insufficient.",
    "The integration of machine learning algorithms has revolutionized data analysis across multiple disciplines.",
    "Moreover, stakeholders across academia and industry must collaborate effectively to address these challenges.",
    "In conclusion, the evidence strongly suggests that comprehensive policy reform is necessary.",
    "The utilization of natural language processing has enhanced our understanding of textual data.",
    "Consequently, future research directions will undoubtedly focus on improving detection accuracy.",
]


def generate(model, text, num_beams=4):
    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=128, num_beams=num_beams, do_sample=False,
            no_repeat_ngram_size=3, early_stopping=True, repetition_penalty=1.2,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def word_change_ratio(orig, mod):
    o = {w.lower().strip(".,;:!?\"'()[]") for w in orig.split() if w.strip()}
    m = {w.lower().strip(".,;:!?\"'()[]") for w in mod.split() if w.strip()}
    if not o: return 0.0
    return 1.0 - len(o & m) / max(len(o), len(m))


print("\n" + "=" * 90)
print("COMPARISON: Original Model vs Fine-Tuned Model")
print("=" * 90)

for i, text in enumerate(test_texts):
    orig_out = generate(orig_model, text)
    fine_out = generate(best_model, text)
    orig_change = word_change_ratio(text, orig_out)
    fine_change = word_change_ratio(text, fine_out)

    print(f"\n\u2500\u2500 Test {i+1} \u2500\u2500")
    print(f"INPUT:     {text}")
    print(f"ORIGINAL:  {orig_out}  [{orig_change:.0%} changed]")
    print(f"FINETUNED: {fine_out}  [{fine_change:.0%} changed]")

# Cleanup original model to free VRAM
del orig_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n" + "=" * 90)
print("\u2705 Testing complete!")

## Step 6b: Test with Your Own Text (Optional)

Paste any AI-generated text below to see how the fine-tuned model rewrites it.

In [ ]:
# ============================
# PASTE YOUR TEXT HERE
# ============================
MY_TEXT = """
The implementation of blockchain technology in supply chain management has demonstrated significant potential for enhancing transparency and traceability. Furthermore, distributed ledger systems facilitate real-time monitoring of goods throughout the logistics pipeline. Moreover, smart contracts automate compliance verification and payment processing. In conclusion, the adoption of blockchain represents a paradigm shift in supply chain operations.
""".strip()

if MY_TEXT:
    # Process sentence by sentence for best results
    sentences = split_sentences(MY_TEXT)
    results = []
    for sent in sentences:
        out = generate(best_model, sent)
        results.append(out)
        print(f"IN:  {sent}")
        print(f"OUT: {out}")
        print()

    full_result = " ".join(results)
    print("\n\u2500\u2500 FULL RESULT \u2500\u2500")
    print(full_result)
    print(f"\nWord change: {word_change_ratio(MY_TEXT, full_result):.0%}")

## Step 7: Save & Download the Fine-Tuned Model

Saves the best checkpoint as the final model and creates a downloadable ZIP.

**After downloading:** Extract the ZIP and replace your local `oxygen-model/` folder contents with the files inside.

In [ ]:
import shutil

OUTPUT_DIR = "/content/oxygen-model-finetuned"

# Save final model
print("Saving final fine-tuned model...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List output files
print(f"\nFiles in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}: {size/1e6:.1f} MB")

# Also save training curves + metadata
import json
metadata = {
    "base_model": "oxygen-model (T5, 248M params)",
    "training_pairs": len(train_pairs),
    "validation_pairs": len(val_pairs),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "best_val_loss": best_val_loss,
    "total_steps": global_step,
    "train_losses": train_losses,
    "val_losses": val_losses,
}
with open(os.path.join(OUTPUT_DIR, "training_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\nTraining metadata saved.")

if os.path.exists("/content/training_curves.png"):
    shutil.copy("/content/training_curves.png", os.path.join(OUTPUT_DIR, "training_curves.png"))

print(f"\n\u2705 Model ready at: {OUTPUT_DIR}/")

In [ ]:
# ── Create ZIP for download ──

ZIP_PATH = "/content/oxygen-model-finetuned"
print("Creating ZIP archive...")
shutil.make_archive(ZIP_PATH, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(f"{ZIP_PATH}.zip") / 1e6
print(f"\u2705 ZIP created: {ZIP_PATH}.zip ({zip_size:.0f} MB)")

In [ ]:
# ── Option A: Download directly (small files only) ──
# Note: 945MB ZIP may timeout in browser. Use Option B for large files.

try:
    from google.colab import files
    files.download(f"{ZIP_PATH}.zip")
except Exception as e:
    print(f"Direct download failed ({e}). Use Option B below.")

In [ ]:
# ── Option B: Copy to Google Drive (recommended for large models) ──

DRIVE_OUTPUT = "/content/drive/MyDrive/oxygen-model-finetuned"

print(f"Copying fine-tuned model to Google Drive...")
print(f"Destination: {DRIVE_OUTPUT}")

if os.path.exists(DRIVE_OUTPUT):
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)

print(f"\n\u2705 Model saved to Google Drive!")
print(f"\nTo use locally:")
print(f"  1. Download the 'oxygen-model-finetuned' folder from Drive")
print(f"  2. Replace contents of your local oxygen-model/ folder")
print(f"  3. Restart oxygen_server.py")

---

## 🎉 Done!

### What to do next:

1. **Download** the fine-tuned model from Google Drive (the `oxygen-model-finetuned` folder)
2. **Replace** the contents of your local `oxygen-model/` folder with the fine-tuned files
3. **Restart** `oxygen_server.py` — it will load the new weights automatically
4. **Test** with your frontend or via curl:
   ```
   curl -X POST http://127.0.0.1:5001/humanize \
     -H "Content-Type: application/json" \
     -d '{"text": "AI has transformed education.", "strength": "medium"}'
   ```

### Tips for better results:
- **More epochs** (5-10) can improve quality at the cost of training time
- **More data**: Add more AI/human pairs to `LOCAL_PAIRS_RAW` in Step 4
- **Lower LR** (1e-5 or 3e-5) if the model overfits (val loss goes up)
- **Larger batch** on A100 GPUs: set `BATCH_SIZE=16` or higher